# Which isoforms move together? WGCNA of transcript isoforms

One objective: find sets of transcript isoforms (modules) whose expression rises and falls together across the samples, and see which modules follow the group. The unit is the measured full-length isoform, so isoforms the long reads discovered (novel) are in the network alongside known ones. This follows the steps of the endotypes-transcriptomics notebooks 12 to 16 (filter, normalize, choose a power, find modules, test modules against the group), so every step has the same source and settings.

In [ ]:
source("src/heatmap.R")
source("src/specific.R")
source("src/wgcna.R")
source("src/paths.R")
need_wgcna()                  # installs WGCNA the first time (a few minutes)
suppressMessages(library(WGCNA))
options(repr.plot.width = 8, repr.plot.height = 5)
loc <- run_locations()        # every folder of the run, from paths.json
comparison <- loc$comparison
min_count <- 10               # an isoform needs at least this many reads ...
max_share <- 0.90             # ... in more than 10% of the samples, or it is left out
min_module_size <- 30         # fewest isoforms in a module
alpha <- 0.05                 # q value below which a module is called associated with the group

**Explanation.** `paths.json` says where the run is, as in the other notebooks. `min_count` and `max_share` are the WGCNA authors' rule for read counts: an isoform with fewer than 10 reads in more than 90% of the samples carries too little information to be placed in a module. `min_module_size` is the smallest number of isoforms WGCNA will call a module (30, the value used in endotypes-transcriptomics). `alpha` is the false discovery rate (q value) used to call a module associated with the group.

In [ ]:
groups <- read_groups(file.path(loc$leafcutter, paste0(comparison, ".lr_leafcutter.groups_file.txt")))
lev <- sort(unique(groups$group))
control_group <- lev[1]; case_group <- lev[2]     # edgeR's order: logFC above 0 means higher in the second group
feat <- read_features("isoforms", file.path(loc$de_transcript, paste0(comparison, "_DTE_edgeR_raw_CPM_matrix.txt")), groups)
counts <- feat$counts
rownames(counts) <- feat$info$id                  # one name per isoform: its id
labels <- setNames(feat$info$label, feat$info$id)
genes  <- setNames(feat$info$gene,  feat$info$id)
c(isoforms = nrow(counts), samples = ncol(counts))
table(groups$group)

**Explanation.** This reads the number of reads for every measured isoform in every sample (the pipeline's `raw_CPM_matrix`, the columns named for the samples). `control_group` and `case_group` are the two groups in the order edgeR used, so a correlation above 0 below means higher in `case_group`, the same way logFC above 0 does in the volcano plot. The table is the number of samples in each group.

In [ ]:
kept <- filter_low_counts(counts, min_count, max_share)
c(isoforms_before = nrow(counts), isoforms_kept = nrow(kept))

**Explanation.** `filter_low_counts()` removes an isoform when fewer than `min_count` reads are found in more than `max_share` of the samples (the WGCNA FAQ rule for RNA-seq counts, used in endotypes-transcriptomics notebook 12). The line shows how many isoforms remain. Long-read data has shallower depth per isoform than short-read data, so this rule may remove many isoforms; if too few remain, lower `min_count` and say so when you present the result.

In [ ]:
E <- pflog1ppf(kept)
datExpr <- t(E)                     # WGCNA wants one row per sample, one column per isoform
gsg <- goodSamplesGenes(datExpr, verbose = 0)
datExpr <- datExpr[gsg$goodSamples, gsg$goodGenes]
c(samples = nrow(datExpr), isoforms = ncol(datExpr), removed_samples = sum(!gsg$goodSamples), removed_isoforms = sum(!gsg$goodGenes))

**Explanation.** `pflog1ppf()` normalizes the reads so that samples with more reads overall are not mistaken for samples with higher expression: scale every sample to the same total (proportional fitting), take log(1 + x), and scale again (PFlog1pPF, Booeshaghi et al. 2022, the method used in endotypes-transcriptomics notebook 12). `goodSamplesGenes()` is WGCNA's own check; it removes isoforms with no variation and samples with too many missing values. The line shows what was left.

In [ ]:
h <- hclust(dist(datExpr), method = "average")
plot(h, labels = groups$group[match(rownames(datExpr), groups$sample)], main = "Sample clustering", xlab = "", sub = "")

**Explanation.** Samples that are alike are joined low in this tree; each is labelled with its group. A sample that hangs far from all the others is an outlier and can shape the modules on its own. Check it before reading the modules; do not remove a sample without a reason you can state.

In [ ]:
pw <- choose_power(datExpr)
pw$fit
c(power = pw$power, rule = pw$rule)

**Explanation.** WGCNA turns correlations into network links by raising them to a power; a higher power keeps only the strongest links. `choose_power()` tries powers 1 to 30 and picks the lowest where the network looks scale-free (R squared at least 0.90, curve flat), as in endotypes-transcriptomics notebook 13. When no power reaches that, it uses the WGCNA FAQ value for a signed network by number of samples (18 for fewer than 20). The last line says which rule chose the power. With few samples the fit is often poor, so the FAQ value is the usual outcome.

In [ ]:
net <- find_modules(datExpr, pw$power, min_module_size)
sizes <- sort(table(net$colors), decreasing = TRUE)
sizes

**Explanation.** `find_modules()` runs WGCNA's `blockwiseModules()` with a signed network, merging modules whose eigengenes correlate above 0.75 (`mergeCutHeight` 0.25), the settings of endotypes-transcriptomics notebook 14. Each isoform gets a module named by a colour. `grey` is the isoforms that fit no module. The table is the number of isoforms in each module.

In [ ]:
ME <- orderMEs(net$MEs)                              # one value per sample for each module: its eigengene
trait <- as.numeric(groups$group[match(rownames(datExpr), groups$sample)] == case_group)
mt <- module_trait_table(ME, trait)
mt$higher_in <- ifelse(mt$r > 0, case_group, control_group)
mt$isoforms <- as.integer(sizes[mt$module])
format(mt, digits = 3)

**Explanation.** A module's *eigengene* is one number per sample that summarises how the whole module behaves there (the first principal component of its isoforms). Each eigengene is correlated with the group (1 for `case_group`, 0 for `control_group`); this is the same as a t test of the eigengene between the groups. `p` is the p value, `q` is the Benjamini-Hochberg correction for the number of modules, and `higher_in` says which group has the higher eigengene. A module is associated with the group when `q` is below `alpha`. With few samples a correlation must be large to reach this, so a module with a small `p` but a larger `q` is a lead, not a finding.

In [ ]:
save_plot(module_trait_plot(mt, alpha, comparison, case_group, control_group), "wgcna_module_group")

**Explanation.** One bar per module, in the module's colour; a bar to the right is higher in `case_group`, to the left higher in `control_group`. A star marks a module with `q` below `alpha`.

In [ ]:
M <- t(ME[, colnames(ME) != "MEgrey", drop = FALSE]); rownames(M) <- sub("^ME", "", rownames(M)); colnames(M) <- rownames(datExpr)
levels_g <- unique(groups$group)
grp <- groups$group[match(colnames(M), groups$sample)]
ht <- Heatmap(M, name = "eigengene",
              col = circlize::colorRamp2(c(-max(abs(M)), 0, max(abs(M))), c("#0072B2", "white", "#D55E00")),
              top_annotation = HeatmapAnnotation(group = grp, col = list(group = group_palette(levels_g)), annotation_name_side = "left"),
              left_annotation = rowAnnotation(module = rownames(M), col = list(module = setNames(rownames(M), rownames(M))), show_legend = FALSE, show_annotation_name = FALSE),
              cluster_rows = FALSE, cluster_columns = TRUE, column_split = factor(grp, levels_g),
              column_title = comparison)
show_heatmap(ht, "wgcna_eigengenes", width = 9, height = 2.5 + 0.3 * nrow(M))

**Explanation.** Each row is a module and each column a sample, split by group. Orange means the module is high in that sample, blue low. A module that follows the group is one colour in one group and the other colour in the other.

## Are the module isoforms known or novel?

In [ ]:
cls_file <- Sys.glob(file.path(loc$sqanti_protein, "*best_ORF_SQANTI_classification.tsv"))[1]
cls <- fread(cls_file, select = c("isoform_id", "tx_cat"), data.table = FALSE)
i <- match(colnames(datExpr), cls$isoform_id)        # NA: the isoform has no predicted ORF
novelty <- ifelse(is.na(i), "no ORF", ifelse(cls$tx_cat[i] == "full-splice_match", "known", "novel"))
by_module <- as.data.frame.matrix(table(module = net$colors, novelty = factor(novelty, c("known", "novel", "no ORF"))))
by_module$percent_novel <- round(100 * by_module$novel / rowSums(by_module[, c("known", "novel", "no ORF")]), 1)
by_module

**Explanation.** For every isoform in the network this takes the SQANTI class from the protein classification file, the same one used in the volcano and heatmap notebooks: **known** is `full-splice_match` (same junctions as a known transcript), **novel** is any other class, and **no ORF** is an isoform without a predicted ORF. The table gives, for each module, how many isoforms are in each class and the percentage that are novel. A module with many novel isoforms that also follows the group is a lead worth following up.

In [ ]:
hubs <- hub_table(datExpr, ME, net$colors, labels, genes, n = 5)
hubs

**Explanation.** A *hub* isoform of a module is one whose expression is most like the module's eigengene (`kME`, a correlation, close to 1 for the most typical isoforms). The five best of each module are listed with their gene. Novel isoforms have no name, so they are shown by their id.

In [ ]:
membership <- data.frame(isoform = colnames(datExpr), gene = genes[colnames(datExpr)], label = labels[colnames(datExpr)],
                         module = net$colors, novelty = novelty, row.names = NULL)
write.table(membership, paste0(figure_stem("wgcna_membership", "tables"), ".tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
write.table(mt,         paste0(figure_stem("wgcna_module_group", "tables"), ".tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
write.table(hubs,       paste0(figure_stem("wgcna_hubs", "tables"), ".tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
write.table(by_module,  paste0(figure_stem("wgcna_module_novelty", "tables"), ".tsv"), sep = "\t", quote = FALSE, row.names = TRUE)

**Explanation.** Four tables are saved in `tables/` with the comparison name: which module every isoform belongs to (with its gene and known or novel class), the module-group correlations, the hub isoforms, and the known/novel counts per module.

## For the bioinformatician

- Input: the sample columns of `*_DTE_edgeR_raw_CPM_matrix.txt` (isoform read counts, not whole numbers because the pipeline estimates them).
- Filter: remove isoforms with fewer than 10 reads in more than 90% of samples (WGCNA FAQ). Normalize: PFlog1pPF (Booeshaghi et al. 2022). Both as in endotypes-transcriptomics notebook 12.
- Network: `pickSoftThreshold` signed, power by the rule of notebook 13; `blockwiseModules` signed network and TOM, `minModuleSize` 30, `mergeCutHeight` 0.25, `maxBlockSize` 5000, fixed seed (notebook 14). Langfelder and Horvath, *BMC Bioinformatics* 2008, doi:10.1186/1471-2105-9-559.
- Module test: Pearson correlation of each eigengene with a 0/1 group (`cor.test`), Benjamini-Hochberg q over modules, as in notebook 16. With under about 20 samples WGCNA modules are unstable; treat the result as exploratory.
- Not done: no module preservation, no correction for sample covariates, and nothing is tested at the isoform level.

In [ ]:
si <- Sys.info()
cat("run on :", si[["nodename"]], "(", si[["sysname"]], si[["release"]], ")\n")
cat("date   :", format(Sys.time(), "%Y-%m-%d %H:%M %Z"), "\n")
cat("R      :", R.version.string, "\n")
for (p in c("data.table", "ComplexHeatmap", "jsonlite", "WGCNA")) cat(sprintf("  %-16s %s\n", p, as.character(packageVersion(p))))

**Explanation.** Records where and with what versions this ran. It is the same in every notebook.